# Segment 8 — LiDAR Validation & SIH Demo

This notebook implements the validation plan from the supplied file: uniform 5 cm vs adaptive 5–50 cm mapping, latency/FPS, memory, cell count, distance-wise analysis, elevation MAE/RMSE, object-preservation tests, visualization, and CSV output. fileciteturn0file0L49-L84

The supplied file is a roadmap rather than source code, so the implementation below is a practical reference implementation.

In [ ]:
import time
import tracemalloc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def load_lidar(path):
    path = Path(path)
    if path.suffix.lower() == ".bin":
        raw = np.fromfile(path, dtype=np.float32)
        if raw.size % 4 != 0:
            raise ValueError("BIN file must contain x,y,z,intensity values.")
        return raw.reshape(-1, 4)
    if path.suffix.lower() == ".npy":
        return np.load(path).astype(np.float32)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path).values.astype(np.float32)
    raise ValueError("Use .bin, .npy or .csv")

def make_demo_lidar(n=120000, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(2, 100, n)
    y = rng.uniform(-25, 25, n)
    z = rng.normal(0, 0.025, n)
    road = np.column_stack([x, y, z])

    objs = []
    for cx, cy, w, d, h in [(18,4,2,4,1.5),(38,-7,1.2,1.2,1.7),
                            (62,9,2.2,4.5,1.6),(78,-3,.8,.8,1.8)]:
        m = 1500
        ox = rng.uniform(cx-w/2, cx+w/2, m)
        oy = rng.uniform(cy-d/2, cy+d/2, m)
        oz = rng.uniform(.05, h, m)
        objs.append(np.column_stack([ox, oy, oz]))
    return np.vstack([road] + objs).astype(np.float32)

# Put your real file here, e.g. "data/000123.bin"
LIDAR_FILE = None

lidar = load_lidar(LIDAR_FILE) if LIDAR_FILE and Path(LIDAR_FILE).exists() else make_demo_lidar()
xyz = lidar[:, :3]
print("LiDAR points:", len(xyz))


## Uniform 5 cm 2.5D map

In [ ]:
def build_uniform_25d(points, resolution=0.05):
    xyz = points[:, :3]
    xmin, ymin = xyz[:,0].min(), xyz[:,1].min()

    ix = np.floor((xyz[:,0] - xmin) / resolution).astype(np.int32)
    iy = np.floor((xyz[:,1] - ymin) / resolution).astype(np.int32)

    keys = np.column_stack((ix, iy))
    unique_keys, inverse = np.unique(keys, axis=0, return_inverse=True)

    z = xyz[:,2]
    zmin = np.full(len(unique_keys), np.inf, dtype=np.float32)
    zmax = np.full(len(unique_keys), -np.inf, dtype=np.float32)
    count = np.bincount(inverse)

    np.minimum.at(zmin, inverse, z)
    np.maximum.at(zmax, inverse, z)

    grid = {
        tuple(k): (float(zmin[i]), float(zmax[i]), int(count[i]))
        for i, k in enumerate(unique_keys)
    }
    return grid


## Adaptive 5–50 cm map

The distance policy follows the supplied plan: 5 cm near the vehicle, then 10 cm, 25 cm and 50 cm farther away. Important-object regions are refined to at least 10 cm. fileciteturn0file0L298-L334

In [ ]:
def adaptive_resolution(x, y, object_mask=None):
    d = np.sqrt(x*x + y*y)

    r = np.where(d <= 10, 0.05,
         np.where(d <= 30, 0.10,
         np.where(d <= 60, 0.25, 0.50)))

    if object_mask is not None:
        r = np.where(object_mask, np.minimum(r, 0.10), r)

    return r

def build_adaptive_25d(points, object_mask=None):
    xyz = points[:, :3]
    x, y, z = xyz.T
    r = adaptive_resolution(x, y, object_mask)

    xmin, ymin = x.min(), y.min()
    ix = np.floor((x-xmin)/r).astype(np.int64)
    iy = np.floor((y-ymin)/r).astype(np.int64)

    cells = {}
    for a, b, rr, zz in zip(ix, iy, r, z):
        key = (int(a), int(b), float(rr))
        if key not in cells:
            cells[key] = [float(zz), float(zz), 1]
        else:
            cells[key][0] = min(cells[key][0], float(zz))
            cells[key][1] = max(cells[key][1], float(zz))
            cells[key][2] += 1

    return cells, r


In [ ]:
def benchmark(method, points, object_mask=None):
    tracemalloc.start()
    t0 = time.perf_counter()

    if method == "Uniform 5cm":
        grid = build_uniform_25d(points)
        resolutions = None
    else:
        grid, resolutions = build_adaptive_25d(points, object_mask)

    latency = (time.perf_counter() - t0) * 1000
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return {
        "method": method,
        "cells": len(grid),
        "latency_ms": latency,
        "fps": 1000 / latency if latency > 0 else np.inf,
        "memory_mb": peak / (1024**2),
        "grid": grid,
        "resolutions": resolutions
    }

# Replace this demo mask with your real Segment 3/4 detector output.
object_mask = (
    ((xyz[:,0] > 16) & (xyz[:,0] < 20) & (xyz[:,1] > 2) & (xyz[:,1] < 6)) |
    ((xyz[:,0] > 37) & (xyz[:,0] < 39) & (xyz[:,1] > -8) & (xyz[:,1] < -6))
)

uniform = benchmark("Uniform 5cm", lidar)
adaptive = benchmark("Adaptive 5-50cm", lidar, object_mask)

results = pd.DataFrame([
    {k:v for k,v in uniform.items() if k not in ["grid","resolutions"]},
    {k:v for k,v in adaptive.items() if k not in ["grid","resolutions"]}
])
results


In [ ]:
def percent_reduction(old, new):
    return (old-new)/old*100 if old else 0

print("Cell reduction:   %.2f%%" % percent_reduction(uniform["cells"], adaptive["cells"]))
print("Memory reduction: %.2f%%" % percent_reduction(uniform["memory_mb"], adaptive["memory_mb"]))
print("Latency reduction: %.2f%%" % percent_reduction(uniform["latency_ms"], adaptive["latency_ms"]))
print("Uniform FPS:      %.2f" % uniform["fps"])
print("Adaptive FPS:     %.2f" % adaptive["fps"])


## Distance-wise analysis

The supplied validation plan uses 0–10 m, 10–30 m, 30–60 m and 60–100 m. fileciteturn0file0L145-L171

In [ ]:
distance = np.sqrt(xyz[:,0]**2 + xyz[:,1]**2)
labels = np.select(
    [distance <= 10, distance <= 30, distance <= 60, distance <= 100],
    ["0-10m","10-30m","30-60m","60-100m"],
    default=">100m"
)

rows = []
for label in ["0-10m","10-30m","30-60m","60-100m"]:
    mask = labels == label
    if mask.any():
        r = benchmark("Adaptive 5-50cm", lidar[mask], object_mask[mask])
        rows.append({
            "distance": label,
            "points": int(mask.sum()),
            "cells": r["cells"],
            "latency_ms": r["latency_ms"],
            "fps": r["fps"],
            "avg_resolution_cm": float(r["resolutions"].mean()*100)
        })

distance_df = pd.DataFrame(rows)
distance_df


## Elevation MAE / RMSE

Use real ground-truth elevations for the final SIH result. The supplied plan explicitly recommends MAE and RMSE. fileciteturn0file0L239-L268

In [ ]:
def elevation_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    err = y_pred[valid] - y_true[valid]
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err**2)))
    }

# Example only — replace with your actual ground truth.
print(elevation_metrics(
    [0.20, 0.30, 0.15, 0.40],
    [0.23, 0.28, 0.18, 0.43]
))


## Important-object test

The supplied plan recommends checking small objects at 20, 40, 60 and 80 m. Connect this table to your real detector/tracker output. fileciteturn0file0L273-L294

In [ ]:
object_test = pd.DataFrame({
    "distance_m": [20, 40, 60, 80],
    "detected": [None, None, None, None],
    "correct_class": [None, None, None, None],
    "position_error_m": [np.nan, np.nan, np.nan, np.nan]
})
object_test


In [ ]:
# Adaptive-resolution visualization
r = adaptive["resolutions"]
plt.figure(figsize=(10,6))
s = plt.scatter(xyz[:,0], xyz[:,1], c=r*100, s=1)
plt.colorbar(s, label="Resolution (cm)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Adaptive 5–50 cm Resolution")
plt.axis("equal")
plt.show()


In [ ]:
# Uniform vs adaptive graphs
for metric, title in [
    ("cells", "Active Cells"),
    ("memory_mb", "Memory (MB)"),
    ("latency_ms", "Latency (ms)"),
    ("fps", "FPS")
]:
    plt.figure(figsize=(7,5))
    plt.bar(["Uniform 5cm", "Adaptive 5-50cm"],
            [uniform[metric], adaptive[metric]])
    plt.ylabel(title)
    plt.title("Uniform vs Adaptive — " + title)
    plt.show()


In [ ]:
# Save results
results.to_csv(RESULTS_DIR/"benchmark.csv", index=False)
distance_df.to_csv(RESULTS_DIR/"distance_analysis.csv", index=False)
object_test.to_csv(RESULTS_DIR/"object_test.csv", index=False)

pd.DataFrame({
    "method": ["Uniform 5cm", "Adaptive 5-50cm"],
    "latency_ms": [uniform["latency_ms"], adaptive["latency_ms"]]
}).to_csv(RESULTS_DIR/"latency.csv", index=False)

pd.DataFrame({
    "method": ["Uniform 5cm", "Adaptive 5-50cm"],
    "memory_mb": [uniform["memory_mb"], adaptive["memory_mb"]]
}).to_csv(RESULTS_DIR/"memory.csv", index=False)

print("Saved results to:", RESULTS_DIR.resolve())


## Final pipeline

LiDAR → preprocessing → AI segmentation → object detection → 2.5D map → adaptive resolution → real-time optimization → validation/dashboard.

The key milestone from the supplied roadmap is one real `.bin` frame successfully passing through classified points, detected objects, a 2.5D map and an adaptive 5–50 cm map. fileciteturn0file0L691-L695